# GISAID Discrepancies

## Housekeeping

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import matplotlib.pyplot as plt

In [2]:
os.chdir("C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/")

gisaid_feb_20_2026 = pd.read_excel("GISAID/downloads/2021-11-01--2026-02-20_Antarctica_North_America_South_America/gisaid_epiflu_isolates.xls")
gisaid_apr_14_2025 = pd.read_csv("GISAID_submission_dates_02-14-2022--12-26-2024_accessed_04-14-2025.csv")
genbank_feb_20_2026 = pd.read_csv("GenBank_submission_dates_02-14-2022--12-26-2024_accessed_02-20-2026.csv")
# genbank_apr_14_2025 = pd.read_csv("GenBank_submission_dates_02-14-2022--12-26-2024_accessed_04-14-2025.csv")

In [3]:
gisaid_apr_14_2025["Submission_Date"] = gisaid_apr_14_2025["Submission_Date"].apply(lambda x: dateutil.parser.parse(x))
gisaid_apr_14_2025["Collection_Date"] = gisaid_apr_14_2025["Collection_Date"].apply(lambda x: dateutil.parser.parse(x))

gisaid_apr_14_2025 = gisaid_apr_14_2025[(gisaid_apr_14_2025["Submission_Date"] >= dateutil.parser.parse("01/01/2023")) & (gisaid_apr_14_2025["Submission_Date"] <= dateutil.parser.parse("09/17/2024"))]
gisaid_apr_14_2025["Isolate"] = gisaid_apr_14_2025["Isolate_Name"].apply(lambda x: x.split("/")[-2])
print(gisaid_apr_14_2025.sort_values("Submission_Date")[["Submission_Date", "Isolate_Name", "Isolate"]])

     Submission_Date                               Isolate_Name        Isolate
991       2023-01-03        A/Striped Skunk/SK/FAV-0824-01/2022    FAV-0824-01
992       2023-01-03              A/Red Fox/SK/FAV-0824-96/2022    FAV-0824-96
993       2023-01-18      A/Canada goose/California/246038/2022         246038
994       2023-01-18     A/eared grebe/North Dakota/245625/2022         245625
995       2023-01-18  A/neotropic cormorant/Arizona/245467/2022         245467
...              ...                                        ...            ...
7368      2024-09-11       A/dairy cow/Idaho/24_024698-002/2024  24_024698-002
7367      2024-09-11       A/dairy cow/Colorado/024240-001/2024     024240-001
7371      2024-09-16    A/dairy cow/Michigan/24_014001-001/2024  24_014001-001
7370      2024-09-16    A/dairy cow/Michigan/24_014001-002/2024  24_014001-002
7369      2024-09-16    A/dairy cow/Michigan/24_014001-004/2024  24_014001-004

[6392 rows x 3 columns]


In [4]:
gisaid_feb_20_2026["Submission_Date"] = gisaid_feb_20_2026["Submission_Date"].apply(lambda x: dateutil.parser.parse(x))
gisaid_feb_20_2026["Collection_Date"] = gisaid_feb_20_2026["Collection_Date"].apply(lambda x: dateutil.parser.parse(x))

gisaid_feb_20_2026 = gisaid_feb_20_2026[(gisaid_feb_20_2026["Submission_Date"] >= dateutil.parser.parse("01/01/2023")) & (gisaid_feb_20_2026["Submission_Date"] <= dateutil.parser.parse("09/17/2024"))]
gisaid_feb_20_2026["Isolate"] = gisaid_feb_20_2026["Isolate_Name"].apply(lambda x: x.split("/")[-2])
gisaid_feb_20_2026["Isolate_gisaid"] = gisaid_feb_20_2026["Isolate"]
print(gisaid_feb_20_2026.sort_values("Submission_Date")[["Submission_Date", "Isolate_Name"]])

      Submission_Date                               Isolate_Name
46         2023-01-03              A/Red Fox/SK/FAV-0824-96/2022
45         2023-01-03        A/Striped Skunk/SK/FAV-0824-01/2022
3530       2023-01-18  A/neotropic cormorant/Arizona/245467/2022
3529       2023-01-18     A/eared grebe/North Dakota/245625/2022
3528       2023-01-18      A/Canada goose/California/246038/2022
...               ...                                        ...
11173      2024-09-11       A/dairy cow/Colorado/024240-001/2024
1698       2024-09-13   A/buff-necked ibis/Bio bio/247636-1/2023
779        2024-09-16    A/dairy cow/Michigan/24_014001-001/2024
778        2024-09-16    A/dairy cow/Michigan/24_014001-002/2024
777        2024-09-16    A/dairy cow/Michigan/24_014001-004/2024

[6859 rows x 2 columns]


In [5]:
# genbank_apr_14_2025["Release_Date"] = genbank_apr_14_2025["Release_Date"].apply(lambda x: dateutil.parser.parse(x))
# genbank_apr_14_2025 = genbank_apr_14_2025[(genbank_apr_14_2025["Release_Date"] >= dateutil.parser.parse("01/01/2023")) & (genbank_apr_14_2025["Release_Date"] <= dateutil.parser.parse("09/17/2024"))]
# print(genbank_apr_14_2025.sort_values("Release_Date")[["Release_Date", "GenBank_Title"]])

In [6]:
genbank_feb_20_2026["Submission_Date"] = genbank_feb_20_2026["Release_Date"].apply(lambda x: dateutil.parser.parse(x))
genbank_feb_20_2026["Collection_Date"] = genbank_feb_20_2026["Collection_Date"].apply(lambda x: dateutil.parser.parse(x))

genbank_feb_20_2026 = genbank_feb_20_2026[(genbank_feb_20_2026["Submission_Date"] >= dateutil.parser.parse("01/01/2023")) & (genbank_feb_20_2026["Submission_Date"] <= dateutil.parser.parse("09/17/2024"))]
genbank_feb_20_2026["Isolate_Name"] = genbank_feb_20_2026["GenBank_Title"].apply(lambda x: x.split("(")[1])
genbank_feb_20_2026["Isolate"] = genbank_feb_20_2026["Isolate_Name"].apply(lambda x: x.split("/")[-2])
genbank_feb_20_2026["Isolate_genbank"] = genbank_feb_20_2026["Isolate"]
print(genbank_feb_20_2026.sort_values("Submission_Date")[["Submission_Date", "Isolate_Name"]])

      Submission_Date                                       Isolate_Name
2069       2023-02-04                      A/gray gull/Chile/C61947/2022
2090       2023-02-04                      A/gray gull/Chile/C61947/2022
2089       2023-02-04                      A/gray gull/Chile/C61947/2022
2088       2023-02-04                      A/gray gull/Chile/C61947/2022
2086       2023-02-04                      A/gray gull/Chile/C61947/2022
...               ...                                                ...
26685      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
26686      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
26687      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
26688      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...
26690      2024-09-17  A/slender-billed parakeet/Araucania /242881-1/...

[24622 rows x 2 columns]


In [7]:
genbank_gisaid_common = genbank_feb_20_2026.merge(gisaid_feb_20_2026, on="Isolate", suffixes=["_genbank", "_gisaid"])
print(genbank_gisaid_common)

       Accession GenBank_RefSeq         Assembly SRA_Accession BioSample  \
0     OQ352538.1        GenBank  GCA_039344385.1           NaN       NaN   
1     OQ352545.1        GenBank  GCA_039343775.1           NaN       NaN   
2     OQ352546.1        GenBank  GCA_039343775.1           NaN       NaN   
3     OQ352547.1        GenBank  GCA_039343775.1           NaN       NaN   
4     OQ352548.1        GenBank  GCA_039343775.1           NaN       NaN   
...          ...            ...              ...           ...       ...   
4950  PQ325300.1        GenBank  GCA_053356475.1           NaN       NaN   
4951  PQ327626.1        GenBank              NaN           NaN       NaN   
4952  PQ327627.1        GenBank              NaN           NaN       NaN   
4953  PQ327628.1        GenBank              NaN           NaN       NaN   
4954  PQ327629.1        GenBank              NaN           NaN       NaN   

        BioProject      Organism_Name                         Species  \
0      PRJNA80

In [8]:
gisaid_time = gisaid_apr_14_2025.merge(gisaid_feb_20_2026, on="Isolate", suffixes=["_old", "_new"])
print(gisaid_time)

        Isolate_Id_old                                 PB2 Segment_Id_old  \
0     EPI_ISL_16367971     EPI2274963|A/Striped Skunk/SK/FAV-0824-01/2022   
1     EPI_ISL_16367899           EPI2274955|A/Red Fox/SK/FAV-0824-96/2022   
2     EPI_ISL_16555205  EPI2298226|PB2_A/Canada goose/California/22-02...   
3     EPI_ISL_16555204  EPI2298218|PB2_A/eared grebe/North Dakota/22-0...   
4     EPI_ISL_16555203  EPI2298210|PB2_A/neotropic cormorant/Arizona/2...   
...                ...                                                ...   
6458  EPI_ISL_19592596  EPI3675337|A/Glaucous-winged gull/Washington/W...   
6459  EPI_ISL_19592595  EPI3675329|A/Glaucous-winged gull/Washington/W...   
6460  EPI_ISL_19592618  EPI3675369|A/Harbor seal/Washington/W232510072...   
6461  EPI_ISL_19592610  EPI3675361|A/Harbor seal/Washington/W232490069...   
6462  EPI_ISL_19592604  EPI3675353|A/Harbor seal/Washington/W232430067...   

                                     PB1 Segment_Id_old  \
0        EPI2274

## Differences between GenBank and GISAID

### Isolate Names

All of these share isolate IDs, but have different full isolate names. This occurs in about half (2719/4955) of our sample. 

Isolate name discrepancy examples:
- GISAID sometimes cuts off or changes the specific type of animal to a general animal (snow goose -> goose, black vulture -> vulture, mallard -> duck, etc.)
- GISAID sometimes subtly changes the location name (Bio Bio -> Biobio, CHL -> Chile, etc.)
- GISAID sometimes changes the specific location to a more general location (Antofagasta -> Chile, etc.)
- Sometimes the isolate ids between genbank and gisaid are the same, but the isolate names are completely different (A/Missouri/121/2024 has the same isolate id as A/red-shouldered hawk/North Carolina/121/2022, etc.)

In [19]:
genbank_gisaid_common[genbank_gisaid_common["Isolate_Name_genbank"] != genbank_gisaid_common["Isolate_Name_gisaid"]][["Isolate_Name_genbank", "Isolate_Name_gisaid"]]

,Isolate_Name_genbank,Isolate_Name_gisaid
16,A/Gull/CHL/227023-3/2022,A/Gull/Chile/227023-3/2022
17,A/Gull/CHL/227023-3/2022,A/Gull/Chile/227023-3/2022
18,A/Gull/CHL/227023-3/2022,A/Gull/Chile/227023-3/2022
19,A/Gull/CHL/227023-2/2022,A/Gull/Chile/227023-2/2022
20,A/Gull/CHL/227023-2/2022,A/Gull/Chile/227023-2/2022
...,...,...
4942,A/gull/Bio Bio/237012/2023,A/gull/Biobio/237012/2023
4951,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022
4952,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022
4953,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022


### Collection Dates

All of these share isolate IDs, but have different collection dates in the metadata. This occurs in (117/4955) entries of our sample.

Date discrepancy examples:
- GISAID sometimes changes the collection date to the beginning of the month (2022-12-26 -> 12-01-2022, etc.)
- GISAID sometimes changes the collection date to a completely different date (2024-03-08 -> 2024-02-26, etc.)
- GISAID sometimes changes the collection date, if unknown and year 2025-2026, to the first of the year (2025 -> 2025-01-01, etc.) (Not in this sample)

In [23]:
genbank_gisaid_common[genbank_gisaid_common["Collection_Date_gisaid"] != genbank_gisaid_common["Collection_Date_genbank"]][["Isolate_Name_genbank", "Isolate_Name_gisaid", "Isolate", "Collection_Date_genbank", "Collection_Date_gisaid"]]

,Isolate_Name_genbank,Isolate_Name_gisaid,Isolate,Collection_Date_genbank,Collection_Date_gisaid
33,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-26,2022-12-01
35,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-26,2022-12-01
37,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-26,2022-12-01
39,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-26,2022-12-01
41,A/Pelecanus/Peru/VFAR-140/2022,A/Pelecanus/Peru/VFAR-140/2022,VFAR-140,2022-12-26,2022-12-01
...,...,...,...,...,...
4652,A/Brown Skua/South Georgia and the South Sandw...,A/dairy cow/Kansas/5/2024,5,2023-10-08,2024-04-26
4951,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,2024-08-22,2022-02-12
4952,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,2024-08-22,2022-02-12
4953,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,2024-08-22,2022-02-12


### Host Names

All of these share isolate IDs, but have different host names in the metadata. This occurs in the **majority** (4811/4955) of our sample, though much of this can be explained by GISAID opting for the common name over the scientific name of each species.

Host name discrepancy examples:
- GISAID often just has the word "Host" in place of a host name (Leucophaeus modestus/gray gull -> Host)
- GISAID often, instead of using the specific species, has the host type in place of a host name (Theristicus caudatus/buff-necked ibis -> Avian)

In [34]:
genbank_gisaid_common[genbank_gisaid_common["Host_gisaid"] != genbank_gisaid_common["Host_genbank"]][["Isolate_Name_genbank", "Isolate_Name_gisaid", "Isolate", "Host_genbank", "Host_gisaid"]]

,Isolate_Name_genbank,Isolate_Name_gisaid,Isolate,Host_genbank,Host_gisaid
0,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
9,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
10,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
11,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
12,A/gray gull/Chile/C61947/2022,A/gray gull/Chile/C61947/2022,C61947,Leucophaeus modestus,Host
...,...,...,...,...,...
4950,A/buff-necked ibis/Bio bio/247636-1/2023,A/buff-necked ibis/Bio bio/247636-1/2023,247636-1,Theristicus caudatus,Avian
4951,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,Homo sapiens,Buteo lineatus
4952,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,Homo sapiens,Buteo lineatus
4953,A/Missouri/121/2024,A/red-shouldered hawk/North Carolina/121/2022,121,Homo sapiens,Buteo lineatus
